## LSTM  Long Short-Term Memory

The aim of this notebook is to show how to train LSTM on a real synthetic dataset.

### The Training Problem

The problem consist in training a LSTM model to predict the next `t` token from a sequence of `n` tokens, generated from a BNF grammar

BNF Definition below:

$$
\begin{array}{rcl}
\langle\mathit{string}\rangle   & \mathrel{::=} & \langle\mathit{term}\rangle \\
                              & \mid          & \langle\mathit{string}\rangle \mathbin{\texttt{+}} \langle\mathit{term}\rangle \\[2pt]
\langle\mathit{term}\rangle   & \mathrel{::=} & AB, ED, K \\[2pt]
\end{array}
$$

The goal is to see if at inference time the model respect the grammar rules and generate valid sequences.

In [46]:
"""Data Generation Utils"""

import numpy as np

ALPHABET = ["A", "B", "D", "E", "K"]
TERMS = ["AB", "ED", "K"]

def generate_seq(length: int) -> str:
    out = ""
    for _ in range(length):
        ri, = np.random.randint(0, len(TERMS), 1, dtype=int)
        out = f"{out}{TERMS[ri]}"

    return out

def generate_seqs(lengths: list[int]) -> list[str]:
    outs = []
    for length in lengths:
        out = ""
        for _ in range(length):
            ri, = np.random.randint(0, len(TERMS), 1, dtype=int)
            out = f"{out}{TERMS[ri]}"
        outs.append(out)
    
    return outs

In [47]:
"""Grammar Parser"""

def parse(input: str) -> bool:
    prev = ""

    for c in input:
        match c:
            case 'A':
                prev = "A"
            case 'B':
                if prev != "A": return False
                prev = ""
            case 'C':
                prev = "C"
            case 'D':
                if prev != "C": return False
                prev = ""
            case 'E':
                continue
            
    return True

## Tokenizer

In order to represent the terms of the language we need to define a tokenizer that will map every letters of the alphabet to point in a vector space.
Our vector space has 4 dimensions:

- Is Vowel
- Is Consonant
- Is Alone (if the terms were letter appear have length 1)
- Position in alphabet

so every term will be represented as a 4D vector in this space.

In [48]:
"""Tokenizer"""

def tokenize(let: str) -> np.ndarray:
    assert len(let) == 1

    is_vowel = False
    is_consonant = False
    is_alone = False
    position = 0

    match let:
        case "A":
            is_vowel = True
            is_consonant = False
            is_alone = False
            position = ALPHABET.index(let)+1
        case "B":
            is_vowel = False
            is_consonant = True
            is_alone = False
            position = ALPHABET.index(let)+1
        case "E":
            is_vowel = True
            is_consonant = False
            is_alone = False
            position = ALPHABET.index(let)+1
        case "D":
            is_vowel = False
            is_consonant = True
            is_alone = False
            position = ALPHABET.index(let)+1
        case "K":
            is_vowel = False
            is_consonant = True
            is_alone = True
            position = ALPHABET.index(let)+1

    return np.array([is_vowel, is_consonant, is_alone, position], dtype=np.float32)

def tokenize_str(input: str) -> np.ndarray:
    out = []
    for let in input:
        out.append(tokenize(let))

    return np.stack(out)

In [49]:
"""Dataset Generation"""

N_SEQ = 50
SPLIT_RATIO = 0.9
TRAIN_LEN = int(N_SEQ*SPLIT_RATIO)
MIN_LENGTH = 5
MAX_LENGTH = 50

random_lengths = np.random.randint(MIN_LENGTH, MAX_LENGTH, N_SEQ)
data = generate_seqs(random_lengths)

train_data, test_data = data[TRAIN_LEN:], data[:TRAIN_LEN]